# STIR-Net V1 — 18 fast screen for event-aware competitive temporal routing

This notebook replaces the expensive **35-step full-scene overfit** as the default development test for the Notebook-17 routing patch.

It asks one narrow question:

> **Given the already learned step-50 upstream representation, can event-aware + competitive fine-node routing quickly create useful split specialization?**

The expensive upstream stack is executed **once** and detached:

```text
Spatial CNN
→ history encoder
→ Detection GNN
→ tracklet pooling
→ CR1
→ CR2
→ spatial decoder to D1
        ↓
   DETACHED CACHE
        ↓
QueryBuilder
→ patched temporal reader
→ QueryDecoder
→ real Hungarian matching
→ real query-bootstrap losses
```

All ~140 queries are retained. Only the expensive upstream recomputation and native-resolution rendering are removed.

## Fast gates

1. **Zero-step causal check** — seconds/minutes.
2. **One backward + optimizer step** — catches autograd/gradient failures.
3. **5-step A/B micro-overfit**
   - baseline independent reader
   - event + competition reader
4. **Same-weight causal ablations** after the 5-step full run.
5. Optional 5→10 step extension only if the result is ambiguous.
6. Optional full integration smoke is disabled by default.

## Reliability

A negative result here is strong evidence to reject/redesign the reader: the modified subsystem cannot solve its own local problem even with a fixed learned upstream representation.

A positive result is **screening evidence**, not final proof. It establishes that the mechanism is learnable and useful enough to justify implementation/integration. Long joint co-adaptation and final native-mask quality still require a later end-to-end acceptance run.

This notebook does **not** modify repository source code.

In [ ]:
from pathlib import Path
from dataclasses import replace, is_dataclass, fields
from types import MethodType
from collections import defaultdict
import copy
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import Tensor, nn
import torch.nn.functional as F

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.query_builder import (
    QUERY_PRIMARY,
    QUERY_SPLIT,
    QUERY_TEMPORAL,
    QUERY_DISCOVERY,
)
from learned.stirnet.model.heads import dot_mask_logits
from learned.stirnet.training.checkpoint import load_checkpoint
from learned.stirnet.training.curriculum import curriculum_stage
from learned.stirnet.training.trainer import move_to_device

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

MICRO_STEPS = 5
EVAL_STEPS = {0, 1, 3, 5}

# Keep these False for the normal fast screen.
RUN_EXTENSION_TO_10 = False
RUN_FULL_INTEGRATION_SMOKE = False
VERIFY_WITH_EXTRA_FULL_FORWARD = False

ROUTING_MODES = {
    "full",
    "no_event",
    "no_competition",
    "baseline",
    "shuffled_event",
}

QUERY_NAMES = {
    QUERY_PRIMARY: "primary",
    QUERY_SPLIT: "split",
    QUERY_TEMPORAL: "temporal",
    QUERY_DISCOVERY: "discovery",
}

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)
NB15_RUN = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "15_hierarchical_temporal_memory"
)
STEP50_CHECKPOINT = NB15_RUN / "checkpoint_temporal_dense.pt"

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "targeted"
    / "18_event_competitive_screen"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 18 requires CUDA.")
if not STEP50_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Missing step-50 checkpoint:\n{STEP50_CHECKPOINT}\n"
        "Run Notebook 15 first."
    )

device = torch.device("cuda")

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("Checkpoint :", STEP50_CHECKPOINT)
print("Micro steps:", MICRO_STEPS)

## 1. Load the same full real scene and construct the event contract

The event signal uses only existing preprocessing features. No GT event label is introduced.

Graph columns:

- `23`: track length before
- `24`: track length after
- `27`: current-frame indicator
- `28`: interior start / newborn-like
- `29`: interior end / broken-like
- `30`: division involvement
- `31`: boundary-related

Continuous tracks remain available.

In [ ]:
batch, sample = build_real_batch(DATA_DIR)
target = batch["targets"][0]

cfg = _reduced_config()
cfg.curriculum.enabled = True
cfg.curriculum.spatial_dense_steps = 30
cfg.curriculum.temporal_dense_steps = 20
cfg.curriculum.query_bootstrap_steps = 20
cfg.curriculum.native_bootstrap_steps = 10

assert sample["current_count"] == 36, sample
assert sample["target_count"] == 33, sample
assert sample["temporal_tracklets"] == 52, sample
assert sample["split_companions_by_source"].get(SOURCE_ID) == 8, sample

N = int(batch["graph_x"].shape[0])
E = int(batch["graph_edge_index"].shape[1])
assert E == N * (N - 1)
assert batch["graph_x"].shape[1] == 32

gx = batch["graph_x"].detach().float().cpu()
window = 2 * int(cfg.temporal.temporal_radius) + 1

EVENT_FEATURES = torch.stack(
    [
        gx[:, 0],
        gx[:, 23] / float(window),
        gx[:, 24] / float(window),
        gx[:, 27],
        gx[:, 28],
        gx[:, 29],
        gx[:, 30],
        gx[:, 31],
    ],
    dim=-1,
).float()

START = gx[:, 28] > 0.5
END = gx[:, 29] > 0.5
DIV = gx[:, 30] > 0.5
BOUNDARY = gx[:, 31] > 0.5

CORRECTION_EVENT = START | END | DIV
CONTINUOUS = (~CORRECTION_EVENT) & (~BOUNDARY)

BASE_EVENT_PRIOR = (
    -1.0
    + 2.0 * START.float()
    + 2.0 * END.float()
    + 1.0 * DIV.float()
    - 0.75 * BOUNDARY.float()
).clamp(-3.0, 3.0)

print("Scene:", sample)
print("nodes / edges / tracklets:", N, E, len(batch["temporal_ref_um"]))

display(pd.DataFrame({
    "category": [
        "interior_start",
        "interior_end",
        "division",
        "correction_event_union",
        "boundary",
        "continuous",
    ],
    "count": [
        int(START.sum()),
        int(END.sum()),
        int(DIV.sum()),
        int(CORRECTION_EVENT.sum()),
        int(BOUNDARY.sum()),
        int(CONTINUOUS.sum()),
    ],
    "fraction": [
        float(START.float().mean()),
        float(END.float().mean()),
        float(DIV.float().mean()),
        float(CORRECTION_EVENT.float().mean()),
        float(BOUNDARY.float().mean()),
        float(CONTINUOUS.float().mean()),
    ],
}))

## 2. Move model inputs to CUDA

In [ ]:
def prepare_device_batch(cpu_batch):
    out = {}
    for key, value in cpu_batch.items():
        if key == "targets":
            out[key] = value
        elif key == "spatial_inputs":
            out[key] = value.to(
                device=device,
                dtype=AMP_DTYPE,
                non_blocking=True,
            )
        elif key == "instance_labels":
            out[key] = value.to(
                device=device,
                dtype=torch.int32,
                non_blocking=True,
            )
        else:
            out[key] = move_to_device(value, device)
    return out

b = prepare_device_batch(batch)

## 3. Notebook-17 patch — fixed autograd-safe version

The source-group competitive update is **out-of-place** (`index_copy`) so the softmax tensor saved by autograd is never mutated.

In [ ]:
class EventRelevance(nn.Module):
    def __init__(self, features, prior, hidden=32):
        super().__init__()
        self.register_buffer(
            "features",
            features.float().clone(),
        )
        self.register_buffer(
            "prior",
            prior.float().clone(),
        )
        self.residual = nn.Sequential(
            nn.Linear(features.shape[-1], hidden),
            nn.SiLU(),
            nn.Linear(hidden, 1),
        )
        nn.init.zeros_(self.residual[-1].weight)
        nn.init.zeros_(self.residual[-1].bias)

    def forward(self):
        residual = self.residual(
            self.features.float()
        ).squeeze(-1)
        logit = (
            self.prior + residual
        ).clamp(-5.0, 5.0)
        return logit, torch.sigmoid(logit), residual


class EventCompetitiveFusion(nn.Module):
    def __init__(
        self,
        base,
        cfg,
        event_features,
        prior,
    ):
        super().__init__()
        self.base = base
        self.cfg = cfg
        self.event = EventRelevance(
            event_features,
            prior,
            int(cfg.relation_bias_hidden),
        )

        self.event_strength_raw = nn.Parameter(
            torch.full(
                (int(cfg.memory_heads),),
                math.log(math.expm1(1.0)),
                dtype=torch.float32,
            )
        )
        self.log_temperature = nn.Parameter(
            torch.tensor(
                math.log(0.5),
                dtype=torch.float32,
            )
        )
        self.mode = "full"

    @property
    def event_strength(self):
        return F.softplus(
            self.event_strength_raw
        ).clamp(0.0, 4.0)

    @property
    def temperature(self):
        return self.log_temperature.exp().clamp(
            0.15,
            2.0,
        )

    def set_mode(self, mode):
        if mode not in ROUTING_MODES:
            raise ValueError(mode)
        self.mode = mode

    def _node_read(
        self,
        norm,
        refs,
        qbatch,
        temporal,
        dref,
        node_tokens,
        source_ids,
        qtypes,
        return_debug,
        full_attention,
    ):
        attn = self.base.node_attention
        mem = temporal.node_memory
        Q = len(norm)
        K = len(node_tokens)

        out = torch.zeros_like(norm)
        entropy = norm.new_zeros(
            Q,
            dtype=torch.float32,
        )
        maxw = norm.new_zeros(
            Q,
            dtype=torch.float32,
        )
        group_size = torch.ones(
            Q,
            device=norm.device,
            dtype=torch.long,
        )

        topk = min(
            max(int(attn.debug_topk), 0),
            K,
        )
        top_indices = torch.full(
            (Q, topk),
            -1,
            device=norm.device,
            dtype=torch.long,
        )
        top_weights = norm.new_zeros(
            (Q, topk),
            dtype=torch.float32,
        )
        full = (
            norm.new_zeros(
                (Q, K),
                dtype=torch.float32,
            )
            if return_debug and full_attention
            else None
        )

        event_logit, event_prob, event_residual = (
            self.event()
        )

        if self.mode == "shuffled_event":
            event_logit = event_logit.roll(1)
            event_prob = event_prob.roll(1)
            event_residual = event_residual.roll(1)

        use_event = self.mode in {
            "full",
            "no_competition",
            "shuffled_event",
        }
        use_competition = self.mode in {
            "full",
            "no_event",
            "shuffled_event",
        }

        for batch_index in torch.unique(
            qbatch
        ).tolist():
            query_ids = torch.nonzero(
                qbatch == batch_index,
                as_tuple=False,
            ).flatten()
            memory_ids = torch.nonzero(
                mem.batch_index == batch_index,
                as_tuple=False,
            ).flatten()

            if (
                query_ids.numel() == 0
                or memory_ids.numel() == 0
            ):
                continue

            qh = attn._split(
                attn.q(norm[query_ids])
            ).permute(1, 0, 2)
            kh = attn._split(
                attn.k(node_tokens[memory_ids])
            ).permute(1, 0, 2)
            vh = attn._split(
                attn.v(node_tokens[memory_ids])
            ).permute(1, 0, 2)

            content = torch.einsum(
                "hqd,hkd->hqk",
                qh,
                kh,
            ).float() / math.sqrt(
                attn.head_dim
            )

            relation = attn.relation_bias(
                refs[query_ids],
                mem.observed_ref_um[memory_ids],
                mem.projected_ref_um[memory_ids],
                mem.time_offset[memory_ids],
                mem.history_valid[memory_ids],
                dref[int(batch_index)],
            ).permute(2, 0, 1).float()

            logits = content + relation

            selection = logits
            if use_event:
                selection = (
                    selection
                    + self.event_strength[
                        :, None, None
                    ]
                    * event_logit[
                        memory_ids
                    ][None, None, :]
                )

            weights = torch.softmax(
                selection,
                dim=-1,
            )

            if (
                use_competition
                and source_ids is not None
                and qtypes is not None
            ):
                local_sources = source_ids[
                    query_ids
                ]
                local_types = qtypes[
                    query_ids
                ]
                seeded = (
                    (local_sources >= 0)
                    & (
                        (local_types == QUERY_PRIMARY)
                        | (local_types == QUERY_SPLIT)
                    )
                )

                for source_id in torch.unique(
                    local_sources[seeded]
                ).tolist():
                    group = torch.nonzero(
                        seeded
                        & (
                            local_sources
                            == source_id
                        ),
                        as_tuple=False,
                    ).flatten()

                    if group.numel() <= 1:
                        continue

                    competition = torch.softmax(
                        logits[
                            :, group, :
                        ]
                        / self.temperature,
                        dim=1,
                    )
                    competitive = (
                        weights[
                            :, group, :
                        ]
                        * competition
                    )
                    competitive = (
                        competitive
                        / competitive.sum(
                            dim=-1,
                            keepdim=True,
                        ).clamp_min(1e-12)
                    )

                    # Autograd-safe: replace the selected query dimension
                    # out-of-place. Never mutate a Softmax output in place.
                    weights = weights.index_copy(
                        1,
                        group,
                        competitive,
                    )
                    group_size[
                        query_ids[group]
                    ] = int(group.numel())

            message = torch.einsum(
                "hqk,hkd->hqd",
                weights,
                vh.float(),
            )
            message = (
                message
                .permute(1, 0, 2)
                .reshape(
                    len(query_ids),
                    attn.d_model,
                )
            )
            out[query_ids] = attn.out(
                message.to(norm.dtype)
            ).to(out.dtype)

            if return_debug:
                mean_weights = weights.mean(
                    dim=0
                )
                entropy[query_ids] = -(
                    mean_weights
                    * mean_weights.clamp_min(
                        1e-12
                    ).log()
                ).sum(dim=-1)
                maxw[query_ids] = (
                    mean_weights.max(
                        dim=-1
                    ).values
                )

                local_topk = min(
                    topk,
                    len(memory_ids),
                )
                if local_topk:
                    local_w, local_i = (
                        torch.topk(
                            mean_weights,
                            local_topk,
                            dim=-1,
                        )
                    )
                    top_indices[
                        query_ids,
                        :local_topk,
                    ] = memory_ids[local_i]
                    top_weights[
                        query_ids,
                        :local_topk,
                    ] = local_w

                if full is not None:
                    full[
                        query_ids[:, None],
                        memory_ids[None, :],
                    ] = mean_weights

        debug = None
        if return_debug:
            debug = {
                "entropy": entropy.detach(),
                "max_weight": maxw.detach(),
                "top_indices": top_indices.detach(),
                "top_weights": top_weights.detach(),
                "query_batch_index": qbatch.detach(),
                "memory_count": torch.tensor(
                    K,
                    device=norm.device,
                ),
                "competition_group_size": group_size.detach(),
                "event_logit": event_logit.detach(),
                "event_probability": event_prob.detach(),
                "event_residual_logit": event_residual.detach(),
                "event_strength": self.event_strength.detach(),
                "competition_temperature": self.temperature.detach(),
                "routing_mode": self.mode,
            }
            if full is not None:
                debug["full_weights"] = (
                    full.detach()
                )

        return out, debug

    def forward(
        self,
        query_tokens,
        query_ref_um,
        query_batch_index,
        temporal,
        dref_um,
        *,
        memory_ablation="full",
        return_debug=False,
        full_attention=False,
        query_source_ids=None,
        query_types=None,
    ):
        norm = self.base.norm(
            query_tokens
        )
        out = query_tokens
        diagnostics = {}
        used = False

        memory = temporal.node_memory

        if (
            memory_ablation != "tracklet_only"
            and memory is not None
            and not memory.is_empty
        ):
            node_tokens = memory.tokens
            if memory_ablation == "zero_node":
                node_tokens = torch.zeros_like(
                    node_tokens
                )
            elif memory_ablation == "shuffle_node":
                node_tokens = (
                    self.base._shuffle_node_tokens(
                        memory
                    )
                )

            node_message, node_debug = (
                self._node_read(
                    norm,
                    query_ref_um,
                    query_batch_index,
                    temporal,
                    dref_um,
                    node_tokens,
                    query_source_ids,
                    query_types,
                    return_debug,
                    full_attention,
                )
            )

            node_gate = torch.sigmoid(
                self.base.node_gate(
                    torch.cat(
                        [norm, node_message],
                        dim=-1,
                    )
                )
            )
            out = (
                out
                + node_gate
                * node_message
            )
            used = True

            if node_debug is not None:
                node_debug[
                    "gate_mean"
                ] = (
                    node_gate.detach()
                    .float()
                    .mean()
                )
                diagnostics[
                    "node"
                ] = node_debug

        if (
            memory_ablation != "node_only"
            and not temporal.is_empty
        ):
            if (
                temporal.history_support_valid
                is not None
                and temporal.history_support_valid.shape[0]
                == temporal.tokens.shape[0]
            ):
                valid = (
                    temporal.history_support_valid.any(
                        dim=-1
                    )
                )
            else:
                valid = torch.ones(
                    len(temporal.tokens),
                    device=temporal.tokens.device,
                    dtype=torch.bool,
                )

            tracklet_message, tracklet_debug = (
                self.base.tracklet_attention(
                    norm,
                    query_ref_um,
                    query_batch_index,
                    temporal.tokens,
                    temporal.ref_um,
                    temporal.ref_um,
                    temporal.ref_um.new_zeros(
                        (len(temporal.tokens),)
                    ),
                    temporal.batch_index,
                    valid,
                    dref_um,
                    return_debug=return_debug,
                    full_attention=full_attention,
                )
            )

            tracklet_gate = torch.sigmoid(
                self.base.tracklet_gate(
                    torch.cat(
                        [
                            norm,
                            tracklet_message,
                        ],
                        dim=-1,
                    )
                )
            )
            out = (
                out
                + tracklet_gate
                * tracklet_message
            )
            used = True

            if tracklet_debug is not None:
                tracklet_debug[
                    "gate_mean"
                ] = (
                    tracklet_gate.detach()
                    .float()
                    .mean()
                )
                diagnostics[
                    "tracklet"
                ] = tracklet_debug

        if used:
            out = (
                out
                + torch.sigmoid(
                    self.base.ffn_gate
                )
                * self.base.ffn(
                    self.base.ffn_norm(out)
                )
            )

        if return_debug:
            diagnostics[
                "routing_mode"
            ] = self.mode

        return (
            out,
            diagnostics
            if return_debug
            else None,
        )

In [ ]:
def patched_layer_forward(
    self,
    q,
    spatial_tokens,
    spatial_pos_um,
    support,
    dref_um,
    spatial_mask_features,
    temporal=None,
    *,
    memory_ablation="full",
    return_debug=False,
    full_attention=False,
):
    x = q.embeddings

    normalized = self.self_norm(x)
    self_message, _ = self.self_attn(
        normalized,
        normalized,
        normalized,
        key_padding_mask=q.padding_mask,
        need_weights=False,
    )
    x = x + self_message

    self.last_temporal_debug = None

    if (
        self.query_memory_enabled
        and temporal is not None
    ):
        valid = ~q.padding_mask
        flat_batch = torch.arange(
            x.shape[0],
            device=x.device,
            dtype=torch.long,
        )[:, None].expand_as(valid)

        temporal_message, self.last_temporal_debug = (
            self.temporal_fusion(
                x[valid],
                (
                    q.references_cellscale
                    * dref_um[
                        :, None, None
                    ]
                )[valid],
                flat_batch[valid],
                temporal,
                dref_um,
                memory_ablation=memory_ablation,
                return_debug=return_debug,
                full_attention=full_attention,
                query_source_ids=q.source_instance_ids[
                    valid
                ],
                query_types=q.query_types[
                    valid
                ],
            )
        )

        if (
            self.last_temporal_debug
            is not None
        ):
            slots = torch.arange(
                x.shape[1],
                device=x.device,
                dtype=torch.long,
            )[None].expand_as(valid)
            self.last_temporal_debug[
                "query_slot_index"
            ] = slots[valid].detach()
            self.last_temporal_debug[
                "query_type"
            ] = q.query_types[
                valid
            ].detach()
            self.last_temporal_debug[
                "source_instance_id"
            ] = q.source_instance_ids[
                valid
            ].detach()

        updated = x.clone()
        updated[valid] = temporal_message
        x = updated

    cross_message = self.cross_attn(
        self.cross_norm(x),
        spatial_tokens,
        spatial_pos_um,
        q.references_cellscale
        * dref_um[:, None, None],
        support,
        q.padding_mask,
        dref_um,
    )
    x = x + cross_message
    x = x + self.ffn(
        self.ffn_norm(x)
    )
    x = x.masked_fill(
        q.padding_mask[..., None],
        0,
    )

    reference_before = (
        q.references_cellscale
    )
    delta = self._bounded_center_delta(
        self.center(x),
        q,
    )
    refs = reference_before + delta

    mask_embedding = self.mask_embed(x)
    masks = dot_mask_logits(
        mask_embedding,
        spatial_mask_features,
    ).masked_fill(
        q.padding_mask[
            ..., None, None, None
        ],
        -20.0,
    )

    return replace(
        q,
        embeddings=x,
        references_cellscale=refs,
    ), {
        "exist_logits": self.exist(x).masked_fill(
            q.padding_mask,
            -20.0,
        ),
        "centers_cellscale": refs,
        "coarse_mask_logits": masks,
        "query_embeddings": x,
        "center_delta_cellscale": delta,
        "reference_before_update_cellscale": reference_before,
    }


def install_patch(model):
    model.query_builder.temporal_fusion = (
        EventCompetitiveFusion(
            model.query_builder.temporal_fusion,
            model.cfg.temporal,
            EVENT_FEATURES,
            BASE_EVENT_PRIOR,
        )
    )

    for layer in model.query_decoder.layers:
        layer.temporal_fusion = (
            EventCompetitiveFusion(
                layer.temporal_fusion,
                model.cfg.temporal,
                EVENT_FEATURES,
                BASE_EVENT_PRIOR,
            )
        )
        layer.forward = MethodType(
            patched_layer_forward,
            layer,
        )

    model.to(device)


def fusions(model):
    return [
        model.query_builder.temporal_fusion,
        *[
            layer.temporal_fusion
            for layer
            in model.query_decoder.layers
        ],
    ]


def set_mode(model, mode):
    for fusion in fusions(model):
        fusion.set_mode(mode)

## 4. Cache the expensive upstream representation **once**

This is the main speedup.

Everything returned from this cell is detached. Query-only optimization cannot backpropagate into the CNN/GNN/CR stack in this screening notebook.

In [ ]:
def detach_tree(value):
    if torch.is_tensor(value):
        return value.detach()
    if is_dataclass(value):
        return type(value)(
            **{
                field.name: detach_tree(
                    getattr(value, field.name)
                )
                for field in fields(value)
            }
        )
    if isinstance(value, dict):
        return {
            key: detach_tree(item)
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [
            detach_tree(item)
            for item in value
        ]
    if isinstance(value, tuple):
        return tuple(
            detach_tree(item)
            for item in value
        )
    return value


@torch.no_grad()
def build_upstream_cache(model, b):
    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        acquisition = model.acquisition(
            b["spacing_um"],
            b["dref_um"],
        )

        pyramid = model.encoder(
            b["spatial_inputs"],
            b["spacing_um"],
            acquisition,
            b.get("spatial_padding_mask"),
        )

        temporal = model._build_temporal(
            b["graph_x"],
            b["graph_edge_index"],
            b["graph_edge_attr"],
            b["tracklet_id"],
            b["temporal_ref_um"],
            b["temporal_status"],
            b["hypothesis_edge_index"],
            b["hypothesis_edge_attr"],
            b["temporal_batch"],
            b["dref_um"],
            node_instance_grid=b.get(
                "node_instance_grid"
            ),
            node_history_valid=b.get(
                "node_history_valid"
            ),
            history_support=b.get(
                "history_support"
            ),
            history_support_valid=b.get(
                "history_support_valid"
            ),
            history_support_dt=b.get(
                "history_support_dt"
            ),
            history_support_center_um=b.get(
                "history_support_center_um"
            ),
            history_support_extent_um=b.get(
                "history_support_extent_um"
            ),
            best_current_component_id=b.get(
                "best_current_component_id"
            ),
            best_component_overlap=b.get(
                "best_component_overlap"
            ),
            second_best_component_overlap=b.get(
                "second_best_component_overlap"
            ),
            node_observed_ref_um=b.get(
                "node_observed_ref_um"
            ),
            node_time_offset=b.get(
                "node_time_offset"
            ),
            node_ids=b.get(
                "node_ids"
            ),
        )

        e3, temporal = model.cr1(
            pyramid.features[3],
            pyramid.spacings_um[3],
            temporal,
            b["dref_um"],
            acquisition,
            (
                pyramid.padding_masks[3]
                if pyramid.padding_masks
                else None
            ),
        )

        e2 = model.decoder.decode_to_e2(
            e3,
            pyramid,
            acquisition,
        )

        e2, temporal = model.cr2(
            e2,
            pyramid.spacings_um[2],
            temporal,
            b["dref_um"],
            acquisition,
            (
                pyramid.padding_masks[2]
                if pyramid.padding_masks
                else None
            ),
        )

        d1, _, _ = (
            model.decoder.decode_from_e2(
                e2,
                pyramid,
                acquisition,
            )
        )

    return {
        "e3": e3.detach(),
        "e2": e2.detach(),
        "d1": d1.detach(),
        "spacings": [
            x.detach()
            for x in (
                pyramid.spacings_um[3],
                pyramid.spacings_um[2],
                pyramid.spacings_um[1],
            )
        ],
        "temporal": detach_tree(
            temporal
        ),
    }


print("Building expensive cache once...")
torch.cuda.reset_peak_memory_stats()
cache_start = time.perf_counter()

cache_model = StirNet(cfg).to(device)
load_checkpoint(
    STEP50_CHECKPOINT,
    cache_model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)
cache_model.eval()

CACHE = build_upstream_cache(
    cache_model,
    b,
)

cache_seconds = (
    time.perf_counter()
    - cache_start
)
cache_peak = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print(
    f"Cache built in {cache_seconds:.2f}s | "
    f"peak allocated {cache_peak:.3f} GiB"
)
print(
    "Cached shapes:",
    "E3", tuple(CACHE["e3"].shape),
    "E2", tuple(CACHE["e2"].shape),
    "D1", tuple(CACHE["d1"].shape),
    "nodes", tuple(
        CACHE["temporal"].node_memory.tokens.shape
    ),
)

del cache_model
gc.collect()
torch.cuda.empty_cache()

print(
    "CUDA allocated after dropping upstream model:",
    f"{torch.cuda.memory_allocated()/1024**3:.3f} GiB",
)

## 5. Cached query-only forward

This uses the actual QueryBuilder and the actual three-layer QueryDecoder.

No query is removed. Therefore decoder self-attention and Hungarian competition still see the full real query set.

In [ ]:
def cached_query_forward(
    model,
    *,
    return_debug=False,
    full_attention=False,
):
    temporal = CACHE["temporal"]

    qstate = model.query_builder(
        CACHE["e2"],
        CACHE["spacings"][1],
        b["instance_labels"],
        b["instance_features"],
        b["instance_ids"],
        b["instance_batch"],
        b["instance_centroids_um"],
        b["dref_um"],
        temporal,
        memory_ablation="full",
        return_debug=return_debug,
        full_attention=full_attention,
    )

    initial_refs = (
        qstate.references_cellscale
    )

    qstate, decoder_outputs = (
        model.query_decoder(
            qstate,
            [
                CACHE["e3"],
                CACHE["e2"],
                CACHE["d1"],
            ],
            CACHE["spacings"],
            b["instance_labels"],
            b["dref_um"],
            temporal,
            memory_ablation="full",
            return_debug=return_debug,
            full_attention=full_attention,
        )
    )

    return (
        qstate,
        decoder_outputs,
        initial_refs,
    )

## 6. Query-bootstrap loss only

This reproduces the **query-dependent part** of the real step-50 curriculum objective:

- existence;
- coarse local Dice;
- coarse local focal;
- center;
- auxiliary decoder-layer query losses;
- real structured Hungarian matching.

The dense foreground/boundary losses are intentionally omitted because their cached values are constant with respect to the subsystem being tested.

Native/count/overlap are already disabled by the real query-bootstrap curriculum.

In [ ]:
QUERY_STAGE = curriculum_stage(
    cfg.curriculum,
    50,
)

print(
    "Targeted curriculum stage:",
    QUERY_STAGE.name,
)
print(
    "Loss overrides:",
    QUERY_STAGE.loss_weight_overrides,
)


def build_query_criterion():
    criterion = RefinementCriterion(
        cfg.losses,
        cfg.queries,
        cfg.training,
    ).to(device)
    criterion.set_loss_weight_overrides(
        QUERY_STAGE.loss_weight_overrides
    )
    return criterion


def query_objective(
    criterion,
    qstate,
    decoder_outputs,
    initial_refs,
):
    final_layer = decoder_outputs[-1]

    final = {
        "exist_logits": final_layer[
            "exist_logits"
        ],
        "centers_cellscale": final_layer[
            "centers_cellscale"
        ],
        "coarse_mask_logits": final_layer[
            "coarse_mask_logits"
        ],
        "coarse_spacing_um": final_layer[
            "coarse_spacing_um"
        ],
        "dref_um": b["dref_um"],
        "query_types": qstate.query_types,
        "source_instance_ids": (
            qstate.source_instance_ids
        ),
        "query_initial_references_cellscale": (
            initial_refs
        ),
    }

    coarse_targets = (
        criterion._coarse_targets(
            final,
            b["targets"],
        )
    )
    matches = criterion._match(
        final,
        qstate.padding_mask,
        b["targets"],
        coarse_targets,
    )

    weights = (
        criterion._effective_loss_weights()
    )
    zero = final[
        "exist_logits"
    ].sum() * 0

    loss_exist = (
        criterion._existence_loss(
            final["exist_logits"],
            qstate.padding_mask,
            matches,
        )
    )
    (
        loss_dice,
        loss_focal,
        loss_center,
    ) = criterion._coarse_losses(
        final,
        matches,
        b["targets"],
        coarse_targets,
    )

    total = (
        weights["exist"] * loss_exist
        + weights["dice_coarse"] * loss_dice
        + weights["focal_coarse"] * loss_focal
        + weights["center"] * loss_center
    )

    aux_total = zero
    if weights["aux_layer"] > 0:
        for aux in decoder_outputs[:-1]:
            aux_for = {
                **aux,
                "dref_um": b["dref_um"],
            }
            aux_targets = (
                criterion._coarse_targets(
                    aux_for,
                    b["targets"],
                )
            )
            aux_exist = (
                criterion._existence_loss(
                    aux["exist_logits"],
                    qstate.padding_mask,
                    matches,
                )
            )
            (
                aux_dice,
                aux_focal,
                aux_center,
            ) = criterion._coarse_losses(
                aux_for,
                matches,
                b["targets"],
                aux_targets,
            )
            aux_total = (
                aux_total
                + weights["aux_layer"]
                * (
                    weights["exist"]
                    * aux_exist
                    + weights[
                        "dice_coarse"
                    ]
                    * aux_dice
                    + weights[
                        "focal_coarse"
                    ]
                    * aux_focal
                    + weights["center"]
                    * aux_center
                )
            )

    total = total + aux_total

    return {
        "loss": total,
        "exist": loss_exist,
        "dice_coarse": loss_dice,
        "focal_coarse": loss_focal,
        "center": loss_center,
        "aux": aux_total,
        "matches": matches,
        "coarse_targets": coarse_targets,
        "final": final,
    }

## 7. Targeted source-9 metrics

The screening metric is **assigned coarse Dice**, not native Dice.

We are testing whether the new temporal reader can produce useful query identities before paying the native-rendering cost.

In [ ]:
def offdiag(matrix):
    if matrix.shape[0] < 2:
        return matrix.new_zeros(0)
    mask = ~torch.eye(
        matrix.shape[0],
        dtype=torch.bool,
        device=matrix.device,
    )
    return matrix[mask]


def source9_indices(qstate):
    query_types = (
        qstate.query_types[0]
        .detach()
        .cpu()
    )
    source_ids = (
        qstate.source_instance_ids[0]
        .detach()
        .cpu()
    )
    seeded = (
        (source_ids == SOURCE_ID)
        & (
            (query_types == QUERY_PRIMARY)
            | (query_types == QUERY_SPLIT)
        )
    )
    indices = torch.nonzero(
        seeded,
        as_tuple=False,
    ).flatten()

    labels = {}
    primary = [
        int(q)
        for q in indices.tolist()
        if int(query_types[q])
        == QUERY_PRIMARY
    ]
    splits = [
        int(q)
        for q in indices.tolist()
        if int(query_types[q])
        == QUERY_SPLIT
    ]
    for q in primary:
        labels[q] = "primary"
    for slot, q in enumerate(splits):
        labels[q] = f"split{slot}"

    return indices, labels


def attention_metrics(model, qstate):
    debug = (
        model.query_decoder.layers[-1]
        .last_temporal_debug
    )
    if (
        debug is None
        or "node" not in debug
        or "full_weights"
        not in debug["node"]
    ):
        raise RuntimeError(
            "Full temporal debug attention "
            "was not requested."
        )

    node_debug = debug["node"]
    slots = (
        debug["query_slot_index"]
        .detach()
        .cpu()
        .long()
    )
    qtypes = (
        debug["query_type"]
        .detach()
        .cpu()
        .long()
    )
    sources = (
        debug["source_instance_id"]
        .detach()
        .cpu()
        .long()
    )

    rows = torch.nonzero(
        (sources == SOURCE_ID)
        & (
            (qtypes == QUERY_PRIMARY)
            | (qtypes == QUERY_SPLIT)
        ),
        as_tuple=False,
    ).flatten()

    weights = (
        node_debug["full_weights"][rows]
        .detach()
        .float()
        .cpu()
    )
    weights = weights / weights.sum(
        dim=-1,
        keepdim=True,
    ).clamp_min(1e-12)

    cosine = (
        F.normalize(weights, dim=-1)
        @ F.normalize(weights, dim=-1).T
    )
    cosine_values = offdiag(
        cosine
    )

    top_nodes = weights.argmax(
        dim=-1
    )
    tracklets = (
        CACHE["temporal"]
        .node_memory
        .tracklet_id
        .detach()
        .cpu()
        .long()
    )

    entropy = -(
        weights
        * weights.clamp_min(1e-12).log()
    ).sum(dim=-1)

    event_mass = weights[
        :, CORRECTION_EVENT
    ].sum(dim=-1)
    continuous_mass = weights[
        :, CONTINUOUS
    ].sum(dim=-1)
    boundary_mass = weights[
        :, BOUNDARY
    ].sum(dim=-1)

    event_fraction = float(
        CORRECTION_EVENT.float().mean()
    )
    continuous_fraction = float(
        CONTINUOUS.float().mean()
    )

    return {
        "attention_cos_mean": float(
            cosine_values.mean()
        ),
        "attention_cos_median": float(
            cosine_values.median()
        ),
        "unique_top_nodes": int(
            torch.unique(top_nodes).numel()
        ),
        "unique_top_tracklets": int(
            torch.unique(
                tracklets[top_nodes]
            ).numel()
        ),
        "normalized_entropy": float(
            (
                entropy
                / math.log(
                    weights.shape[-1]
                )
            ).mean()
        ),
        "mean_max_weight": float(
            weights.max(dim=-1).values.mean()
        ),
        "correction_event_mass": float(
            event_mass.mean()
        ),
        "continuous_mass": float(
            continuous_mass.mean()
        ),
        "boundary_mass": float(
            boundary_mass.mean()
        ),
        "event_enrichment": float(
            event_mass.mean()
            / max(event_fraction, 1e-8)
        ),
        "continuous_enrichment": float(
            continuous_mass.mean()
            / max(
                continuous_fraction,
                1e-8,
            )
        ),
        "competition_group_size": float(
            node_debug[
                "competition_group_size"
            ][rows]
            .detach()
            .float()
            .mean()
            .cpu()
        ),
    }


def assigned_coarse_metrics(
    qstate,
    objective,
):
    indices, labels = source9_indices(
        qstate
    )
    match = objective["matches"][0]
    mapping = {
        int(q): int(t)
        for q, t in zip(
            match.pred_indices.detach().cpu(),
            match.target_indices.detach().cpu(),
        )
    }

    gt_ids = target[
        "ids"
    ].detach().cpu()

    prediction = (
        objective["final"][
            "coarse_mask_logits"
        ][0]
        .float()
        .sigmoid()
    )
    coarse_targets = (
        objective["coarse_targets"][0]
        .float()
    )

    rows = []
    dice_values = []

    for q in indices.tolist():
        target_index = mapping.get(
            int(q)
        )
        if target_index is None:
            rows.append({
                "query": int(q),
                "query_label": labels[int(q)],
                "gt_id": -1,
                "assigned_coarse_dice": float("nan"),
            })
            continue

        pred = prediction[int(q)].flatten()
        gt = coarse_targets[
            target_index
        ].flatten()

        dice = (
            2 * (pred * gt).sum()
            + 1e-6
        ) / (
            pred.sum()
            + gt.sum()
            + 1e-6
        )
        dice = float(
            dice.detach().cpu()
        )
        dice_values.append(dice)

        rows.append({
            "query": int(q),
            "query_label": labels[int(q)],
            "gt_id": int(
                gt_ids[target_index]
            ),
            "assigned_coarse_dice": dice,
        })

    return (
        {
            "assigned_coarse_dice_mean": (
                float(np.mean(dice_values))
                if dice_values
                else float("nan")
            ),
            "source9_matched": int(
                len(dice_values)
            ),
        },
        pd.DataFrame(rows),
    )


def center_metrics(qstate):
    indices, _ = source9_indices(
        qstate
    )
    centers_um = (
        qstate.references_cellscale[
            0,
            indices.to(
                qstate.references_cellscale.device
            ),
        ]
        .detach()
        .float()
        .cpu()
        * float(
            b["dref_um"][0]
            .detach()
            .cpu()
        )
    )

    pair = torch.pdist(
        centers_um
    )
    return {
        "center_pair_mean_um": (
            float(pair.mean())
            if pair.numel()
            else float("nan")
        ),
        "center_pair_median_um": (
            float(pair.median())
            if pair.numel()
            else float("nan")
        ),
        "center_pair_max_um": (
            float(pair.max())
            if pair.numel()
            else float("nan")
        ),
    }

## 8. Branch construction and fast evaluation

In [ ]:
def build_branch(mode):
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    model = StirNet(cfg).to(device)

    load_checkpoint(
        STEP50_CHECKPOINT,
        model,
        optimizer=None,
        scheduler=None,
        scaler=None,
        map_location="cpu",
        strict=True,
        migrate_history=True,
    )

    install_patch(model)
    set_mode(model, mode)

    # Deterministic micro-screen: gradients are enabled, but dropout is
    # disabled. This reduces A/B noise and tests representational capacity.
    model.eval()

    criterion = (
        build_query_criterion()
    )

    query_parameters = list(
        model.query_builder.parameters()
    ) + list(
        model.query_decoder.parameters()
    )

    optimizer = torch.optim.AdamW(
        query_parameters,
        lr=float(cfg.training.lr),
        weight_decay=float(
            cfg.training.weight_decay
        ),
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
        init_scale=1024.0,
    )

    return (
        model,
        criterion,
        optimizer,
        scaler,
    )


@torch.no_grad()
def evaluate_branch(
    model,
    criterion,
    *,
    tag,
    micro_step,
):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        (
            qstate,
            decoder_outputs,
            initial_refs,
        ) = cached_query_forward(
            model,
            return_debug=True,
            full_attention=True,
        )
        objective = query_objective(
            criterion,
            qstate,
            decoder_outputs,
            initial_refs,
        )

    attention = attention_metrics(
        model,
        qstate,
    )
    assigned, assignment_table = (
        assigned_coarse_metrics(
            qstate,
            objective,
        )
    )
    centers = center_metrics(qstate)

    summary = {
        "tag": tag,
        "micro_step": int(
            micro_step
        ),
        "routing_mode": (
            model.query_decoder
            .layers[-1]
            .temporal_fusion.mode
        ),
        "query_loss": float(
            objective["loss"]
            .detach()
            .cpu()
        ),
        "exist": float(
            objective["exist"]
            .detach()
            .cpu()
        ),
        "dice_coarse_loss": float(
            objective["dice_coarse"]
            .detach()
            .cpu()
        ),
        "focal_coarse": float(
            objective["focal_coarse"]
            .detach()
            .cpu()
        ),
        "center_loss": float(
            objective["center"]
            .detach()
            .cpu()
        ),
        "matched_gt": int(
            objective["matches"][0]
            .target_indices
            .numel()
        ),
        "elapsed_s": float(
            time.perf_counter()
            - start
        ),
        "peak_cuda_gib": float(
            torch.cuda.max_memory_allocated()
            / 1024**3
        ),
        **attention,
        **assigned,
        **centers,
    }

    return {
        "summary": summary,
        "assignment_table": assignment_table,
    }

# Gate 1 — zero-step causal screen

All rows use the same step-50 weights and the same cached upstream tensors.

This should take only query-path forwards.

In [ ]:
gate1_model, gate1_criterion, _, _ = (
    build_branch("baseline")
)

gate1_rows = []
gate1_details = {}

for mode in [
    "baseline",
    "full",
    "no_event",
    "no_competition",
    "shuffled_event",
]:
    set_mode(
        gate1_model,
        mode,
    )
    result = evaluate_branch(
        gate1_model,
        gate1_criterion,
        tag=f"zero_{mode}",
        micro_step=0,
    )
    gate1_rows.append(
        result["summary"]
    )
    gate1_details[mode] = result

gate1_df = pd.DataFrame(
    gate1_rows
)
gate1_df.to_csv(
    RUN_DIR
    / "gate1_zero_step_ablations.csv",
    index=False,
)

display(gate1_df[[
    "routing_mode",
    "query_loss",
    "correction_event_mass",
    "continuous_mass",
    "event_enrichment",
    "continuous_enrichment",
    "attention_cos_mean",
    "unique_top_nodes",
    "unique_top_tracklets",
    "center_pair_median_um",
    "assigned_coarse_dice_mean",
    "elapsed_s",
]])

del gate1_model, gate1_criterion
gc.collect()
torch.cuda.empty_cache()

# Gate 2 — one backward + optimizer step

This gate exists specifically to catch:

- in-place autograd failures;
- zero/disconnected gradients;
- NaN/Inf gradients;
- OOM;
- optimizer-step failures.

The scratch model is discarded afterward, so the A/B comparison still starts from identical step-50 weights.

In [ ]:
def patch_gradient_report(model):
    rows = []

    for fusion_name, fusion in zip(
        [
            "component",
            "decoder0",
            "decoder1",
            "decoder2",
        ],
        fusions(model),
    ):
        parameter_map = {
            "event_strength": (
                fusion.event_strength_raw
            ),
            "competition_temperature": (
                fusion.log_temperature
            ),
            "event_residual_last_weight": (
                fusion.event.residual[-1].weight
            ),
            "event_residual_last_bias": (
                fusion.event.residual[-1].bias
            ),
        }

        for name, parameter in (
            parameter_map.items()
        ):
            grad = parameter.grad
            rows.append({
                "fusion": fusion_name,
                "parameter": name,
                "has_grad": grad is not None,
                "grad_finite": (
                    bool(
                        torch.isfinite(
                            grad
                        ).all()
                    )
                    if grad is not None
                    else False
                ),
                "grad_abs_sum": (
                    float(
                        grad.detach()
                        .float()
                        .abs()
                        .sum()
                        .cpu()
                    )
                    if grad is not None
                    else 0.0
                ),
            })

    return pd.DataFrame(rows)


scratch_model, scratch_criterion, scratch_optimizer, scratch_scaler = (
    build_branch("full")
)

scratch_optimizer.zero_grad(
    set_to_none=True
)

torch.cuda.reset_peak_memory_stats()
gate2_start = time.perf_counter()

with torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    (
        qstate,
        decoder_outputs,
        initial_refs,
    ) = cached_query_forward(
        scratch_model,
        return_debug=False,
        full_attention=False,
    )
    objective = query_objective(
        scratch_criterion,
        qstate,
        decoder_outputs,
        initial_refs,
    )

loss_before = float(
    objective["loss"]
    .detach()
    .cpu()
)

scratch_scaler.scale(
    objective["loss"]
).backward()
scratch_scaler.unscale_(
    scratch_optimizer
)

grad_df = patch_gradient_report(
    scratch_model
)

all_query_grads = [
    parameter.grad
    for parameter
    in list(
        scratch_model.query_builder.parameters()
    )
    + list(
        scratch_model.query_decoder.parameters()
    )
    if parameter.grad is not None
]

query_grad_finite = all(
    bool(torch.isfinite(g).all())
    for g in all_query_grads
)
query_grad_nonzero = sum(
    float(
        g.detach()
        .float()
        .abs()
        .sum()
        .cpu()
    )
    for g in all_query_grads
)

torch.nn.utils.clip_grad_norm_(
    list(
        scratch_model.query_builder.parameters()
    )
    + list(
        scratch_model.query_decoder.parameters()
    ),
    float(
        cfg.training.max_grad_norm
    ),
)

scratch_scaler.step(
    scratch_optimizer
)
scratch_scaler.update()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    (
        qstate_after,
        decoder_after,
        initial_after,
    ) = cached_query_forward(
        scratch_model,
        return_debug=False,
        full_attention=False,
    )
    objective_after = query_objective(
        scratch_criterion,
        qstate_after,
        decoder_after,
        initial_after,
    )

loss_after = float(
    objective_after["loss"]
    .detach()
    .cpu()
)

gate2_summary = pd.DataFrame([{
    "loss_before": loss_before,
    "loss_after_one_step": loss_after,
    "query_grad_finite": query_grad_finite,
    "query_grad_abs_sum": query_grad_nonzero,
    "elapsed_s": float(
        time.perf_counter()
        - gate2_start
    ),
    "peak_cuda_gib": float(
        torch.cuda.max_memory_allocated()
        / 1024**3
    ),
}])

display(gate2_summary)
display(grad_df)

grad_df.to_csv(
    RUN_DIR
    / "gate2_patch_gradients.csv",
    index=False,
)

if not query_grad_finite:
    raise RuntimeError(
        "Gate 2 failed: non-finite query gradients."
    )
if query_grad_nonzero <= 0:
    raise RuntimeError(
        "Gate 2 failed: all query gradients are zero."
    )

# Competition does not exist at component-level query building:
# split siblings have not been constructed yet. Therefore the component
# fusion's competition-temperature parameter is intentionally unused and
# should have grad=None. Requiring a gradient there would make Gate 2 fail
# even when autograd is completely healthy.

expected_patch_grad = grad_df[
    (
        (grad_df["parameter"] == "event_strength")
        |
        (
            (grad_df["parameter"] == "competition_temperature")
            & (grad_df["fusion"] != "component")
        )
    )
].copy()

bad_expected = expected_patch_grad[
    (~expected_patch_grad["has_grad"])
    | (~expected_patch_grad["grad_finite"])
]

if len(bad_expected):
    display(bad_expected)
    raise RuntimeError(
        "Gate 2 failed: an expected event/competition routing gradient "
        "is missing or non-finite."
    )

# A completely zero routing gradient is also a real failure.
if float(expected_patch_grad["grad_abs_sum"].sum()) <= 0:
    raise RuntimeError(
        "Gate 2 failed: all expected routing gradients are zero."
    )

component_competition = grad_df[
    (grad_df["fusion"] == "component")
    & (grad_df["parameter"] == "competition_temperature")
]
if len(component_competition):
    print(
        "Expected unused parameter: component competition_temperature | "
        f"has_grad={bool(component_competition.iloc[0]['has_grad'])}"
    )

print("GATE 2 PASSED.")

del (
    scratch_model,
    scratch_criterion,
    scratch_optimizer,
    scratch_scaler,
    qstate,
    decoder_outputs,
    objective,
    qstate_after,
    decoder_after,
    objective_after,
)
gc.collect()
torch.cuda.empty_cache()

# Gate 3 — 5-step A/B micro-overfit

Two branches start from the exact same step-50 checkpoint:

- `baseline`: old independent temporal reader;
- `full`: event prioritization + source-group competition.

The upstream cache is identical for both.

Evaluation is performed at micro-steps `0, 1, 3, 5`.

In [ ]:
def micro_train_step(
    model,
    criterion,
    optimizer,
    scaler,
):
    optimizer.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        (
            qstate,
            decoder_outputs,
            initial_refs,
        ) = cached_query_forward(
            model,
            return_debug=False,
            full_attention=False,
        )
        objective = query_objective(
            criterion,
            qstate,
            decoder_outputs,
            initial_refs,
        )

    scaler.scale(
        objective["loss"]
    ).backward()
    scaler.unscale_(
        optimizer
    )

    parameters = (
        list(
            model.query_builder.parameters()
        )
        + list(
            model.query_decoder.parameters()
        )
    )

    torch.nn.utils.clip_grad_norm_(
        parameters,
        float(
            cfg.training.max_grad_norm
        ),
    )

    scaler.step(
        optimizer
    )
    scaler.update()

    return {
        "loss": float(
            objective["loss"]
            .detach()
            .cpu()
        ),
        "dice_coarse_loss": float(
            objective["dice_coarse"]
            .detach()
            .cpu()
        ),
        "center_loss": float(
            objective["center"]
            .detach()
            .cpu()
        ),
    }


def run_micro_branch(
    mode,
    *,
    steps=MICRO_STEPS,
):
    model, criterion, optimizer, scaler = (
        build_branch(mode)
    )

    rows = []
    assignments = []
    train_rows = []

    if 0 in EVAL_STEPS:
        evaluation = evaluate_branch(
            model,
            criterion,
            tag=f"{mode}_step0",
            micro_step=0,
        )
        rows.append(
            evaluation["summary"]
        )
        table = evaluation[
            "assignment_table"
        ].copy()
        table.insert(
            0,
            "micro_step",
            0,
        )
        assignments.append(table)

    start = time.perf_counter()

    for step in range(
        1,
        steps + 1,
    ):
        train_metric = micro_train_step(
            model,
            criterion,
            optimizer,
            scaler,
        )
        train_rows.append({
            "routing_mode": mode,
            "micro_step": step,
            **train_metric,
        })

        if step in EVAL_STEPS:
            evaluation = evaluate_branch(
                model,
                criterion,
                tag=f"{mode}_step{step}",
                micro_step=step,
            )
            rows.append(
                evaluation["summary"]
            )

            table = evaluation[
                "assignment_table"
            ].copy()
            table.insert(
                0,
                "micro_step",
                step,
            )
            assignments.append(
                table
            )

            print(
                f"{mode:8s} step {step}: "
                f"loss={evaluation['summary']['query_loss']:.4f} | "
                f"event={evaluation['summary']['event_enrichment']:.2f}x | "
                f"attn_cos={evaluation['summary']['attention_cos_mean']:.6f} | "
                f"top_nodes={evaluation['summary']['unique_top_nodes']} | "
                f"center_med={evaluation['summary']['center_pair_median_um']:.4f}um | "
                f"assigned_coarse={evaluation['summary']['assigned_coarse_dice_mean']:.4f}"
            )

    elapsed = (
        time.perf_counter()
        - start
    )

    return {
        "model": model,
        "criterion": criterion,
        "optimizer": optimizer,
        "scaler": scaler,
        "metrics": pd.DataFrame(rows),
        "assignments": pd.concat(
            assignments,
            ignore_index=True,
        ),
        "training": pd.DataFrame(
            train_rows
        ),
        "elapsed_s": elapsed,
    }


print("Running BASELINE branch...")
baseline_run = run_micro_branch(
    "baseline"
)

print("\nRunning FULL patch branch...")
full_run = run_micro_branch(
    "full"
)

micro_df = pd.concat(
    [
        baseline_run["metrics"],
        full_run["metrics"],
    ],
    ignore_index=True,
)

micro_df.to_csv(
    RUN_DIR
    / "gate3_micro_ab_metrics.csv",
    index=False,
)

display(micro_df[[
    "routing_mode",
    "micro_step",
    "query_loss",
    "correction_event_mass",
    "continuous_mass",
    "event_enrichment",
    "continuous_enrichment",
    "attention_cos_mean",
    "unique_top_nodes",
    "unique_top_tracklets",
    "normalized_entropy",
    "center_pair_median_um",
    "assigned_coarse_dice_mean",
    "matched_gt",
    "elapsed_s",
]])

print(
    "Baseline branch training seconds:",
    round(
        baseline_run["elapsed_s"],
        2,
    ),
)
print(
    "Full branch training seconds:",
    round(
        full_run["elapsed_s"],
        2,
    ),
)

## 9. Hungarian assignment stability over the micro-run

Different attention is only useful if query identity does not become arbitrary.

In [ ]:
def assignment_churn(
    table,
    mode,
):
    rows = []

    for label, group in table.groupby(
        "query_label"
    ):
        ordered = group.sort_values(
            "micro_step"
        )
        gt = ordered[
            "gt_id"
        ].to_numpy()
        switches = (
            int(
                np.count_nonzero(
                    gt[1:] != gt[:-1]
                )
            )
            if len(gt) > 1
            else 0
        )
        rows.append({
            "routing_mode": mode,
            "query_label": label,
            "observations": len(group),
            "unique_targets": int(
                group["gt_id"].nunique()
            ),
            "switches": switches,
            "final_gt_id": int(
                ordered.iloc[-1][
                    "gt_id"
                ]
            ),
        })

    return pd.DataFrame(rows)


churn_df = pd.concat(
    [
        assignment_churn(
            baseline_run[
                "assignments"
            ],
            "baseline",
        ),
        assignment_churn(
            full_run[
                "assignments"
            ],
            "full",
        ),
    ],
    ignore_index=True,
)

churn_df.to_csv(
    RUN_DIR
    / "gate3_assignment_churn.csv",
    index=False,
)

display(churn_df)

# Gate 4 — same-weight causal ablations after the 5-step full run

The same trained `full` query-path weights are evaluated under:

- full
- no event
- no competition
- baseline reader
- shuffled event identities

This is much stronger than independently training five models.

In [ ]:
trained_model = full_run[
    "model"
]
trained_criterion = full_run[
    "criterion"
]

ablation_rows = []
ablation_details = {}

for mode in [
    "full",
    "no_event",
    "no_competition",
    "baseline",
    "shuffled_event",
]:
    set_mode(
        trained_model,
        mode,
    )

    result = evaluate_branch(
        trained_model,
        trained_criterion,
        tag=f"same_weight_{mode}",
        micro_step=MICRO_STEPS,
    )

    ablation_rows.append(
        result["summary"]
    )
    ablation_details[
        mode
    ] = result

ablation_df = pd.DataFrame(
    ablation_rows
)

ablation_df.to_csv(
    RUN_DIR
    / "gate4_same_weight_ablations.csv",
    index=False,
)

display(ablation_df[[
    "routing_mode",
    "query_loss",
    "correction_event_mass",
    "continuous_mass",
    "event_enrichment",
    "attention_cos_mean",
    "unique_top_nodes",
    "unique_top_tracklets",
    "center_pair_median_um",
    "assigned_coarse_dice_mean",
]])

# Gate 5 — automatic RED / YELLOW / GREEN screen

The automatic classification is deliberately conservative.

### RED
The reader still fails to produce multiple temporal explanations after five steps.

### YELLOW
The mechanism diversifies attention, but biological usefulness is not yet established.

### GREEN
The full branch:
- remains event-focused;
- produces multiple temporal explanations;
- is more diverse than baseline;
- does not reduce assigned coarse Dice;
- and preferably improves assigned coarse Dice / geometry.

A GREEN result means **worth implementing/integrating**, not "final model solved."

In [ ]:
def final_row(
    dataframe,
    mode,
):
    candidates = dataframe[
        dataframe["routing_mode"]
        == mode
    ]
    return candidates.sort_values(
        "micro_step"
    ).iloc[-1]


baseline_final = final_row(
    micro_df,
    "baseline",
)
full_final = final_row(
    micro_df,
    "full",
)

same_full = ablation_df[
    ablation_df["routing_mode"]
    == "full"
].iloc[0]
same_shuffled = ablation_df[
    ablation_df["routing_mode"]
    == "shuffled_event"
].iloc[0]
same_no_event = ablation_df[
    ablation_df["routing_mode"]
    == "no_event"
].iloc[0]
same_no_comp = ablation_df[
    ablation_df["routing_mode"]
    == "no_competition"
].iloc[0]

event_focus = (
    full_final[
        "event_enrichment"
    ] > 1.5
    and full_final[
        "continuous_enrichment"
    ] < 0.75
)

diversified = (
    int(
        full_final[
            "unique_top_nodes"
        ]
    ) >= 2
    and full_final[
        "attention_cos_mean"
    ]
    < baseline_final[
        "attention_cos_mean"
    ] - 1e-4
)

useful = (
    full_final[
        "assigned_coarse_dice_mean"
    ]
    >= baseline_final[
        "assigned_coarse_dice_mean"
    ] - 0.002
)

improved_task_signal = (
    full_final[
        "assigned_coarse_dice_mean"
    ]
    > baseline_final[
        "assigned_coarse_dice_mean"
    ] + 0.002
    or full_final[
        "center_pair_median_um"
    ]
    > baseline_final[
        "center_pair_median_um"
    ] + 0.01
)

identity_sensitive = (
    same_full[
        "assigned_coarse_dice_mean"
    ]
    >= same_shuffled[
        "assigned_coarse_dice_mean"
    ] - 0.002
)

if not diversified:
    verdict = "RED"
    reason = (
        "After five cached query steps, "
        "source-9 hypotheses still do not "
        "form multiple temporal explanations."
    )
elif (
    event_focus
    and useful
    and improved_task_signal
):
    verdict = "GREEN"
    reason = (
        "The patch produces event-focused, "
        "diverse temporal explanations without "
        "hurting assigned coarse Dice and shows "
        "a useful task/geometry improvement."
    )
else:
    verdict = "YELLOW"
    reason = (
        "The reader shows some intended mechanism "
        "but task usefulness is not yet strong enough. "
        "Use the optional 5→10 cached extension before "
        "considering any full overfit."
    )

report = {
    "verdict": verdict,
    "reason": reason,
    "event_focus": bool(event_focus),
    "diversified": bool(diversified),
    "useful": bool(useful),
    "improved_task_signal": bool(
        improved_task_signal
    ),
    "identity_sensitive_noninferior": bool(
        identity_sensitive
    ),
    "baseline_final": {
        key: (
            value.item()
            if hasattr(
                value,
                "item",
            )
            else value
        )
        for key, value
        in baseline_final.to_dict().items()
    },
    "full_final": {
        key: (
            value.item()
            if hasattr(
                value,
                "item",
            )
            else value
        )
        for key, value
        in full_final.to_dict().items()
    },
}

print("=" * 50)
print("NOTEBOOK 18 VERDICT:", verdict)
print(reason)
print("=" * 50)
print(
    "event_focus           :", event_focus
)
print(
    "diversified           :", diversified
)
print(
    "useful/non-inferior   :", useful
)
print(
    "task/geometry improved:",
    improved_task_signal
)

with (
    RUN_DIR
    / "screening_verdict.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        report,
        handle,
        indent=2,
        default=float,
    )

## 11. Visualize the fast A/B trajectory

In [ ]:
plot_metrics = [
    "attention_cos_mean",
    "unique_top_nodes",
    "center_pair_median_um",
    "assigned_coarse_dice_mean",
]

for metric in plot_metrics:
    pivot = micro_df.pivot(
        index="micro_step",
        columns="routing_mode",
        values=metric,
    )
    ax = pivot.plot(
        marker="o",
        figsize=(7, 3.5),
    )
    ax.set_title(metric)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

# Optional Gate 5b — extend only if YELLOW

Default: **disabled**.

If the five-step result is YELLOW, set `RUN_EXTENSION_TO_10 = True` and run this cell. It continues only the already-trained full branch for five more cached query steps.

Do not extend a RED result merely because "more training might help."

In [ ]:
if RUN_EXTENSION_TO_10:
    set_mode(
        trained_model,
        "full",
    )

    extension_rows = []

    for step in range(
        MICRO_STEPS + 1,
        11,
    ):
        train_metric = micro_train_step(
            trained_model,
            trained_criterion,
            full_run["optimizer"],
            full_run["scaler"],
        )

        if step in {6, 8, 10}:
            evaluation = evaluate_branch(
                trained_model,
                trained_criterion,
                tag=f"full_step{step}",
                micro_step=step,
            )
            extension_rows.append(
                evaluation["summary"]
            )
            print(
                f"full step {step}: "
                f"attn_cos={evaluation['summary']['attention_cos_mean']:.6f} | "
                f"top_nodes={evaluation['summary']['unique_top_nodes']} | "
                f"center_med={evaluation['summary']['center_pair_median_um']:.4f}um | "
                f"assigned_coarse={evaluation['summary']['assigned_coarse_dice_mean']:.4f}"
            )

    extension_df = pd.DataFrame(
        extension_rows
    )
    extension_df.to_csv(
        RUN_DIR
        / "optional_extension_to_10.csv",
        index=False,
    )
    display(extension_df)
else:
    print(
        "Extension disabled. "
        "Enable only for a YELLOW result."
    )

# Optional Gate 6 — one real full-model integration smoke

Default: **disabled**.

This is not part of the normal fast screen. It exists only after a GREEN result to verify that the patched query modules still execute inside a genuine full forward/backward graph.

It intentionally performs **one** expensive step, not a 35-step overfit.

In [ ]:
if RUN_FULL_INTEGRATION_SMOKE:
    print(
        "Running one expensive full integration step..."
    )

    # The current trained_model contains the micro-trained query modules.
    set_mode(
        trained_model,
        "full",
    )
    trained_model.train()

    full_optimizer = torch.optim.AdamW(
        list(
            trained_model.query_builder.parameters()
        )
        + list(
            trained_model.query_decoder.parameters()
        ),
        lr=float(cfg.training.lr),
        weight_decay=float(
            cfg.training.weight_decay
        ),
    )
    full_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )

    full_optimizer.zero_grad(
        set_to_none=True
    )
    torch.cuda.reset_peak_memory_stats()
    integration_start = time.perf_counter()

    # Full graph, but only query-dependent objective is backpropagated.
    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        # Reconstruct the full upstream graph without caching.
        acquisition = trained_model.acquisition(
            b["spacing_um"],
            b["dref_um"],
        )
        pyramid = trained_model.encoder(
            b["spatial_inputs"],
            b["spacing_um"],
            acquisition,
            b.get("spatial_padding_mask"),
        )
        temporal = trained_model._build_temporal(
            b["graph_x"],
            b["graph_edge_index"],
            b["graph_edge_attr"],
            b["tracklet_id"],
            b["temporal_ref_um"],
            b["temporal_status"],
            b["hypothesis_edge_index"],
            b["hypothesis_edge_attr"],
            b["temporal_batch"],
            b["dref_um"],
            node_instance_grid=b.get("node_instance_grid"),
            node_history_valid=b.get("node_history_valid"),
            history_support=b.get("history_support"),
            history_support_valid=b.get("history_support_valid"),
            history_support_dt=b.get("history_support_dt"),
            history_support_center_um=b.get("history_support_center_um"),
            history_support_extent_um=b.get("history_support_extent_um"),
            best_current_component_id=b.get("best_current_component_id"),
            best_component_overlap=b.get("best_component_overlap"),
            second_best_component_overlap=b.get("second_best_component_overlap"),
            node_observed_ref_um=b.get("node_observed_ref_um"),
            node_time_offset=b.get("node_time_offset"),
            node_ids=b.get("node_ids"),
        )
        e3, temporal = trained_model.cr1(
            pyramid.features[3],
            pyramid.spacings_um[3],
            temporal,
            b["dref_um"],
            acquisition,
            pyramid.padding_masks[3]
            if pyramid.padding_masks
            else None,
        )
        e2 = trained_model.decoder.decode_to_e2(
            e3,
            pyramid,
            acquisition,
        )
        e2, temporal = trained_model.cr2(
            e2,
            pyramid.spacings_um[2],
            temporal,
            b["dref_um"],
            acquisition,
            pyramid.padding_masks[2]
            if pyramid.padding_masks
            else None,
        )
        d1, _, _ = trained_model.decoder.decode_from_e2(
            e2,
            pyramid,
            acquisition,
        )

        qstate = trained_model.query_builder(
            e2,
            pyramid.spacings_um[2],
            b["instance_labels"],
            b["instance_features"],
            b["instance_ids"],
            b["instance_batch"],
            b["instance_centroids_um"],
            b["dref_um"],
            temporal,
        )
        initial_refs = qstate.references_cellscale
        qstate, decoder_outputs = trained_model.query_decoder(
            qstate,
            [e3, e2, d1],
            [
                pyramid.spacings_um[3],
                pyramid.spacings_um[2],
                pyramid.spacings_um[1],
            ],
            b["instance_labels"],
            b["dref_um"],
            temporal,
        )
        objective = query_objective(
            trained_criterion,
            qstate,
            decoder_outputs,
            initial_refs,
        )

    full_scaler.scale(
        objective["loss"]
    ).backward()
    full_scaler.unscale_(
        full_optimizer
    )

    finite = all(
        torch.isfinite(p.grad).all()
        for p in (
            list(
                trained_model.query_builder.parameters()
            )
            + list(
                trained_model.query_decoder.parameters()
            )
        )
        if p.grad is not None
    )

    print(
        "Full integration loss:",
        float(
            objective["loss"]
            .detach()
            .cpu()
        ),
    )
    print(
        "Gradients finite:",
        bool(finite),
    )
    print(
        "Elapsed:",
        round(
            time.perf_counter()
            - integration_start,
            2,
        ),
        "s",
    )
    print(
        "Peak CUDA:",
        round(
            torch.cuda.max_memory_allocated()
            / 1024**3,
            3,
        ),
        "GiB",
    )
else:
    print(
        "Full integration smoke disabled."
    )

## 13. Save compact artifacts and timing summary

In [ ]:
baseline_run["assignments"].to_csv(
    RUN_DIR
    / "baseline_assignment_history.csv",
    index=False,
)
full_run["assignments"].to_csv(
    RUN_DIR
    / "full_assignment_history.csv",
    index=False,
)
baseline_run["training"].to_csv(
    RUN_DIR
    / "baseline_training.csv",
    index=False,
)
full_run["training"].to_csv(
    RUN_DIR
    / "full_training.csv",
    index=False,
)

timing = {
    "upstream_cache_seconds": cache_seconds,
    "baseline_micro_training_seconds": (
        baseline_run["elapsed_s"]
    ),
    "full_micro_training_seconds": (
        full_run["elapsed_s"]
    ),
    "total_screen_seconds_excluding_optional": (
        cache_seconds
        + baseline_run["elapsed_s"]
        + full_run["elapsed_s"]
    ),
}

with (
    RUN_DIR / "timing.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        timing,
        handle,
        indent=2,
    )

print(json.dumps(
    timing,
    indent=2,
))

print("\nArtifacts:")
for path in sorted(
    RUN_DIR.glob("*")
):
    print(" ", path.name)

set_mode(
    trained_model,
    "full",
)
gc.collect()
torch.cuda.empty_cache()
print("CUDA cache cleared.")

# How to use Notebook 18 in the development loop

## RED

Stop. Do not run a full overfit.

The modified reader failed its own local mechanism test.

## YELLOW

Do **not** run the full overfit.

Run only the optional cached extension to step 10. If the mechanism still does not translate into useful assigned coarse Dice/geometry, redesign.

## GREEN

The patch has passed a high-value subsystem screen:

- gradients work;
- event information is actually used;
- sibling temporal explanations diversify;
- the result is not obviously biologically worse.

At that point, implement the change cleanly or run the optional one-step integration smoke.

A full multi-stage overfit should be reserved for a later **acceptance checkpoint after several targeted changes have already passed their own screens**.